In [9]:
import pandas as pd

submission = pd.read_csv("../data/submission_template.csv")
landsat_val = pd.read_csv("../data/landsat_features_validation.csv")
terraclimate_val = pd.read_csv("../data/terraclimate_features_validation.csv")

In [10]:
submission["Sample Date"] = pd.to_datetime(submission["Sample Date"], dayfirst=True)
landsat_val["Sample Date"] = pd.to_datetime(landsat_val["Sample Date"], dayfirst=True)
terraclimate_val["Sample Date"] = pd.to_datetime(terraclimate_val["Sample Date"], dayfirst=True)

In [11]:
submission["month"] = submission["Sample Date"].dt.month
submission["year"] = submission["Sample Date"].dt.year
submission["dayofyear"] = submission["Sample Date"].dt.dayofyear

In [12]:
df_val = submission.merge(
    landsat_val,
    on=["Latitude","Longitude","Sample Date"],
    how="left"
)

df_val = df_val.merge(
    terraclimate_val,
    on=["Latitude","Longitude","Sample Date"],
    how="left"
)

In [13]:
df_val["nir_swir16_ratio"] = df_val["nir"] / df_val["swir16"]
df_val["nir_swir22_ratio"] = df_val["nir"] / df_val["swir22"]
df_val["green_nir_ratio"] = df_val["green"] / df_val["nir"]

df_val["nir_minus_swir16"] = df_val["nir"] - df_val["swir16"]
df_val["nir_minus_green"] = df_val["nir"] - df_val["green"]

df_val["ndmi_pet"] = df_val["NDMI"] * df_val["pet"]
df_val["mndwi_pet"] = df_val["MNDWI"] * df_val["pet"]

df_val["swir_ratio"] = df_val["swir16"] / df_val["swir22"]

In [14]:
df_val.fillna(df_val.median(numeric_only=True), inplace=True)

In [15]:
X_val = df_val.drop(columns=["Sample Date","Total Alkalinity","Electrical Conductance","Dissolved Reactive Phosphorus"], errors="ignore")

In [19]:
X_val = df_val.drop(
    columns=[
        "Sample Date",
        "Total Alkalinity",
        "Electrical Conductance",
        "Dissolved Reactive Phosphorus"
    ],
    errors="ignore"
)

In [23]:
from sklearn.ensemble import RandomForestRegressor

In [24]:
water_quality = pd.read_csv("../data/water_quality_training_dataset.csv")
landsat = pd.read_csv("../data/landsat_features_training.csv")
terraclimate = pd.read_csv("../data/terraclimate_features_training.csv")

water_quality["Sample Date"] = pd.to_datetime(water_quality["Sample Date"], dayfirst=True)
landsat["Sample Date"] = pd.to_datetime(landsat["Sample Date"], dayfirst=True)
terraclimate["Sample Date"] = pd.to_datetime(terraclimate["Sample Date"], dayfirst=True)

water_quality["month"] = water_quality["Sample Date"].dt.month
water_quality["year"] = water_quality["Sample Date"].dt.year
water_quality["dayofyear"] = water_quality["Sample Date"].dt.dayofyear

df = water_quality.merge(
    landsat,
    on=["Latitude","Longitude","Sample Date"],
    how="left"
)

df = df.merge(
    terraclimate,
    on=["Latitude","Longitude","Sample Date"],
    how="left"
)

df["nir_swir16_ratio"] = df["nir"] / df["swir16"]
df["nir_swir22_ratio"] = df["nir"] / df["swir22"]
df["green_nir_ratio"] = df["green"] / df["nir"]

df["nir_minus_swir16"] = df["nir"] - df["swir16"]
df["nir_minus_green"] = df["nir"] - df["green"]

df["ndmi_pet"] = df["NDMI"] * df["pet"]
df["mndwi_pet"] = df["MNDWI"] * df["pet"]

df["swir_ratio"] = df["swir16"] / df["swir22"]

df.fillna(df.median(numeric_only=True), inplace=True)

targets = [
    "Total Alkalinity",
    "Electrical Conductance",
    "Dissolved Reactive Phosphorus"
]

X = df.drop(columns=targets + ["Sample Date"])
y = df[targets]

In [25]:
rf_final = RandomForestRegressor(
    n_estimators=200,
    random_state=42
)

rf_final.fit(X, y)

RandomForestRegressor(n_estimators=200, random_state=42)

In [26]:
predictions = rf_final.predict(X_val)

In [32]:
submission["Total Alkalinity"] = predictions[:, 0]
submission["Electrical Conductance"] = predictions[:, 1]
submission["Dissolved Reactive Phosphorus"] = predictions[:, 2]

In [33]:
submission.shape


(200, 9)

In [34]:
submission.head()

,Latitude,Longitude,Sample Date,Total Alkalinity,Electrical Conductance,Dissolved Reactive Phosphorus,month,year,dayofyear
0,-32.043333,27.822778,2014-09-01,133.985185,299.91785,23.510,9,2014,244
1,-33.329167,26.077500,2015-09-16,190.436955,530.49645,67.380,9,2015,259
2,-32.991639,27.640028,2015-05-07,102.343485,236.79470,18.435,5,2015,127
3,-34.096389,24.439167,2012-02-07,173.782320,544.00835,68.385,2,2012,38
4,-32.000556,28.581667,2014-10-01,88.349390,242.98110,23.945,10,2014,274


In [35]:
submission_v1 = submission[
    [
        "Longitude",
        "Latitude",
        "Sample Date",
        "Total Alkalinity",
        "Electrical Conductance",
        "Dissolved Reactive Phosphorus"
    ]
]

In [36]:
submission_v1.to_csv("../submissions/submission_v1.csv", index=False)

In [38]:
submission_v1.head()

,Longitude,Latitude,Sample Date,Total Alkalinity,Electrical Conductance,Dissolved Reactive Phosphorus
0,27.822778,-32.043333,2014-09-01,133.985185,299.91785,23.510
1,26.077500,-33.329167,2015-09-16,190.436955,530.49645,67.380
2,27.640028,-32.991639,2015-05-07,102.343485,236.79470,18.435
3,24.439167,-34.096389,2012-02-07,173.782320,544.00835,68.385
4,28.581667,-32.000556,2014-10-01,88.349390,242.98110,23.945


In [39]:
submission_v1.shape

(200, 6)